[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/10.%20The%20Dual-Cycling%20Quay%20Crane%20Problem/P10-Tier-2_executed.ipynb) [![Open In SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open In Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 10. The Dual-Cycling Quay Crane Problem

## Tier 2 — A Divide-and-Conquer Heuristic

### Goal

This notebook builds a heuristic schedule for quay-crane operations using spatial decomposition, an adapted Johnson sequencing rule, and a coordination step. It represents a dual cycle as a paired import-unload and export-load operation, with a modeled reduction in the paired loading time. The method is a heuristic; it does not prove that the resulting schedule is optimal.

### Inputs and outputs

The example uses six vessel bays, three stacks per bay, three cranes, and generated import and export containers. The data-generation seed is fixed for reproducibility. The solver produces an operation schedule, a makespan in minutes, a count and ratio of dual cycles, and decomposition and coordination statistics. A simple round-robin schedule is included as a reference comparison.

### Method outline

1. Group stacks into adjacent-bay subproblems and assign one crane to each group.
2. If a subproblem exceeds the stack threshold, partition its stacks into hatch-blocked and accessible groups.
3. Sequence each group using Johnson's rule applied to each stack's total unload and load times.
4. Build container-level timelines, pairing as many imports and exports within a stack as possible.
5. Identify overlapping operations in adjacent bays and apply a schedule-wide delay to the later crane as a coordination heuristic.
6. Inspect adjacent unload-load operations for additional dual-cycle opportunities and report the final schedule.

### Model assumptions and scope

Processing times are deterministic once the example data have been generated. Each stack is processed as a unit for sequencing, with its imports unloaded before its exports are loaded. A dual cycle pairs one import operation with one export operation in the same stack, up to the smaller of the two container counts. Each paired export load is modeled as taking 80% of its nominal load time. The 20% saving is a modeling assumption, not a measured operational result.

Bays are represented by integer indices, and potential interference is approximated by temporal overlap between operations in adjacent bays. The coordination step is a simplified delay heuristic; it is not a detailed crane-motion model and does not certify that every physical conflict has been eliminated. The example does not model crane travel distances, detailed hatch-access precedence, vessel stability, labor constraints, or stochastic processing times. Results therefore illustrate the method and its assumptions rather than establish operational feasibility or real-time performance.

In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# A fixed seed makes the generated example repeatable.
np.random.seed(42)
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

### Algorithm foundation

**Intuition.** Decomposition reduces the size of each sequencing task: stacks in nearby bays are grouped, and larger groups can be separated by accessibility. Johnson's rule then gives an ordering based on two processing times for each stack. Coordination considers overlaps between the schedules produced for different crane groups.

**Johnson's rule.** For each stack \(j\), let \(U_j\) be its total unloading time and \(L_j\) its total loading time. Repeatedly select the remaining stack with the smallest value of \(\min(U_j,L_j)\). If \(U_j \leq L_j\), place it in the earliest available position; otherwise, place it in the latest available position. The resulting sequence is a Johnson ordering for a two-stage flow-shop representation. Here it is used as a sequencing heuristic for the stack-level dual-cycling model; the notebook does not claim that the full quay-crane problem satisfies all assumptions required for a global optimality guarantee from classical Johnson's rule.

The decomposition is spatial: stacks are grouped by bay, and a large group is split into accessible and hatch-blocked stacks. A stack is a base case when its group contains no more than the configured threshold. The example's paired-bay groups contain six stacks, so they meet the default threshold directly. The program partitions a larger group once; it does not recursively subdivide the resulting groups.

In [ ]:
@dataclass
class ContainerStack:
    """A stack containing import and export containers."""

    id: str
    bay: int
    position: int
    import_containers: List[Dict]
    export_containers: List[Dict]
    total_unload_time: float
    total_load_time: float
    hatch_blocked: bool = False

    @property
    def min_time(self) -> float:
        """Return the smaller stage time used by Johnson's rule."""
        return min(self.total_unload_time, self.total_load_time)

    @property
    def total_containers(self) -> int:
        return len(self.import_containers) + len(self.export_containers)


@dataclass
class BaySubproblem:
    """A stack group assigned to a crane for local sequencing."""

    id: str
    bays: List[int]
    stacks: List[ContainerStack]
    assigned_crane: Optional[int] = None
    processing_time: float = 0.0
    dual_cycles: int = 0


@dataclass
class CoordinationPoint:
    """An overlapping interval identified for coordination."""

    time_range: Tuple[float, float]
    involved_cranes: List[int]
    conflict_type: str
    resolution_cost: float


@dataclass
class DualCycleSchedule:
    """A schedule and summary measures for crane operations."""

    crane_operations: Dict[int, List[Dict]]
    makespan: float
    dual_cycles: int
    total_operations: int
    coordination_overhead: float

    @property
    def dual_cycle_ratio(self) -> float:
        """Return the fraction of operations covered by paired cycles."""
        if self.total_operations == 0:
            return 0.0
        return 2 * self.dual_cycles / self.total_operations


print("Data structures defined successfully")

In [ ]:
class DualCycleDivideConquer:
    """Construct a divide-and-conquer heuristic schedule."""

    def __init__(
        self,
        num_cranes: int = 3,
        num_bays: int = 6,
        stacks_per_bay: int = 3,
        threshold: int = 6,
    ):
        if (num_cranes, num_bays, stacks_per_bay) != (3, 6, 3):
            raise ValueError(
                "The supplied example data require 3 cranes, 6 bays, "
                "and 3 stacks per bay."
            )
        if threshold < 1:
            raise ValueError("The decomposition threshold must be positive.")

        self.num_cranes = num_cranes
        self.num_bays = num_bays
        self.stacks_per_bay = stacks_per_bay
        self.threshold = threshold
        self.coordination_delay = 3.0
        self.decomposition_stats = {
            "tree_depth": 0,
            "subproblems_generated": 0,
            "coordination_points": 0,
            "johnson_applications": 0,
        }
        self.initialize_problem()

    def initialize_problem(self):
        """Generate the fixed-size example and its processing times."""
        self.stacks = []
        stack_id = 0
        bay_times = {
            1: [12, 8, 15],
            2: [10, 14, 9],
            3: [11, 13, 7],
            4: [16, 6, 12],
            5: [9, 18, 10],
            6: [14, 8, 11],
        }

        for bay in range(1, self.num_bays + 1):
            for position in range(self.stacks_per_bay):
                unload_time = bay_times[bay][position]
                load_time = np.random.uniform(5, 10)
                num_import = np.random.randint(2, 5)
                num_export = np.random.randint(2, 4)

                import_containers = [
                    {
                        "id": f"I{stack_id}_{i}",
                        "weight": np.random.uniform(10, 25),
                        "unload_time": unload_time / num_import,
                    }
                    for i in range(num_import)
                ]
                export_containers = [
                    {
                        "id": f"E{stack_id}_{i}",
                        "weight": np.random.uniform(8, 20),
                        "load_time": load_time / num_export,
                    }
                    for i in range(num_export)
                ]

                stack = ContainerStack(
                    id=f"S{stack_id}",
                    bay=bay,
                    position=position,
                    import_containers=import_containers,
                    export_containers=export_containers,
                    total_unload_time=unload_time,
                    total_load_time=load_time,
                    hatch_blocked=(bay == 1 and position >= 1),
                )
                self.stacks.append(stack)
                stack_id += 1

        print(f"Initialized {len(self.stacks)} stacks across {self.num_bays} bays")
        print(
            "Total import containers: "
            f"{sum(len(stack.import_containers) for stack in self.stacks)}"
        )
        print(
            "Total export containers: "
            f"{sum(len(stack.export_containers) for stack in self.stacks)}"
        )

    def decompose_by_bays(
        self, stacks: List[ContainerStack]
    ) -> List[BaySubproblem]:
        """Group stacks into the example's adjacent-bay subproblems."""
        bay_stacks = defaultdict(list)
        for stack in stacks:
            bay_stacks[stack.bay].append(stack)

        bay_pairs = [(1, 2), (3, 4), (5, 6)]
        subproblems = []
        for crane_id, (bay1, bay2) in enumerate(bay_pairs):
            subproblem = BaySubproblem(
                id=f"Bay_{bay1}-{bay2}",
                bays=[bay1, bay2],
                stacks=bay_stacks[bay1] + bay_stacks[bay2],
                assigned_crane=crane_id,
            )
            subproblems.append(subproblem)
            self.decomposition_stats["subproblems_generated"] += 1
        return subproblems

    def partition_stacks(self, bay: BaySubproblem) -> List[BaySubproblem]:
        """Split a large group by hatch accessibility."""
        if len(bay.stacks) <= self.threshold:
            return [bay]

        blocked = [stack for stack in bay.stacks if stack.hatch_blocked]
        accessible = [stack for stack in bay.stacks if not stack.hatch_blocked]
        groups = []
        for suffix, stacks in (("Blocked", blocked), ("Accessible", accessible)):
            if stacks:
                groups.append(
                    BaySubproblem(
                        id=f"{bay.id}_{suffix}",
                        bays=bay.bays,
                        stacks=stacks,
                        assigned_crane=bay.assigned_crane,
                    )
                )
                self.decomposition_stats["subproblems_generated"] += 1
        return groups

    def johnsons_rule_dual_cycle(
        self, bay: BaySubproblem
    ) -> DualCycleSchedule:
        """Order stacks with the adapted Johnson sequencing rule."""
        self.decomposition_stats["johnson_applications"] += 1
        remaining = bay.stacks.copy()
        front = []
        back = []

        while remaining:
            min_stack = min(remaining, key=lambda stack: stack.min_time)
            remaining.remove(min_stack)
            if min_stack.total_unload_time <= min_stack.total_load_time:
                front.append(min_stack)
            else:
                back.insert(0, min_stack)

        sequence = front + back
        return self.create_dual_cycle_schedule(sequence, bay)

    def create_dual_cycle_schedule(
        self, sequence: List[ContainerStack], bay: BaySubproblem
    ) -> DualCycleSchedule:
        """Build a timeline, pairing imports and exports within each stack."""
        crane_id = bay.assigned_crane
        if crane_id is None:
            raise ValueError("Each subproblem must have an assigned crane.")

        operations = {crane_id: []}
        current_time = 0.0
        dual_cycles = 0
        total_operations = 0

        for stack in sequence:
            paired_exports = min(
                len(stack.import_containers), len(stack.export_containers)
            )
            for container in stack.import_containers:
                duration = container["unload_time"]
                operations[crane_id].append(
                    {
                        "type": "unload",
                        "container_id": container["id"],
                        "stack_id": stack.id,
                        "bay": stack.bay,
                        "start_time": current_time,
                        "duration": duration,
                        "end_time": current_time + duration,
                        "dual_cycle": False,
                    }
                )
                current_time += duration
                total_operations += 1

            for index, container in enumerate(stack.export_containers):
                is_dual_cycle = index < paired_exports
                nominal_duration = container["load_time"]
                duration = nominal_duration * (0.8 if is_dual_cycle else 1.0)
                operations[crane_id].append(
                    {
                        "type": "load",
                        "container_id": container["id"],
                        "stack_id": stack.id,
                        "bay": stack.bay,
                        "start_time": current_time,
                        "duration": duration,
                        "end_time": current_time + duration,
                        "dual_cycle": is_dual_cycle,
                    }
                )
                current_time += duration
                total_operations += 1
                dual_cycles += int(is_dual_cycle)

        makespan = max(
            (op["end_time"] for ops in operations.values() for op in ops),
            default=0.0,
        )
        return DualCycleSchedule(
            crane_operations=operations,
            makespan=makespan,
            dual_cycles=dual_cycles,
            total_operations=total_operations,
            coordination_overhead=0.0,
        )

    def identify_interference_points(
        self, bay_solutions: List[DualCycleSchedule]
    ) -> List[CoordinationPoint]:
        """Find time overlaps between operations in adjacent bays."""
        points = []
        for index, solution_i in enumerate(bay_solutions):
            for solution_j in bay_solutions[index + 1 :]:
                for crane_i, ops_i in solution_i.crane_operations.items():
                    for crane_j, ops_j in solution_j.crane_operations.items():
                        if crane_i == crane_j:
                            continue
                        for op_i in ops_i:
                            for op_j in ops_j:
                                start = max(op_i["start_time"], op_j["start_time"])
                                end = min(op_i["end_time"], op_j["end_time"])
                                adjacent_bays = abs(op_i["bay"] - op_j["bay"]) <= 1
                                if start < end and adjacent_bays:
                                    points.append(
                                        CoordinationPoint(
                                            time_range=(start, end),
                                            involved_cranes=[crane_i, crane_j],
                                            conflict_type="interference",
                                            resolution_cost=self.coordination_delay,
                                        )
                                    )
                                    self.decomposition_stats["coordination_points"] += 1
        return points

    def solve_coordination_subproblem(
        self,
        cranes: List[int],
        time_range: Tuple[float, float],
    ) -> float:
        """Return a conservative delay for an identified overlap."""
        overlap_length = max(0.0, time_range[1] - time_range[0])
        return overlap_length + self.coordination_delay

    def coordinate_bay_solutions(
        self, bay_solutions: List[DualCycleSchedule]
    ) -> DualCycleSchedule:
        """Apply schedule-wide offsets for detected inter-crane overlaps."""
        points = self.identify_interference_points(bay_solutions)
        total_coordination_overhead = 0.0

        for point in points:
            resolution_cost = self.solve_coordination_subproblem(
                point.involved_cranes, point.time_range
            )
            total_coordination_overhead += resolution_cost
            delayed_crane = point.involved_cranes[1]
            for solution in bay_solutions:
                operations = solution.crane_operations.get(delayed_crane)
                if operations is not None:
                    for operation in operations:
                        operation["start_time"] += resolution_cost
                        operation["end_time"] += resolution_cost

        merged_operations = defaultdict(list)
        total_dual_cycles = 0
        total_operations = 0
        for solution in bay_solutions:
            for crane_id, operations in solution.crane_operations.items():
                merged_operations[crane_id].extend(operations)
            total_dual_cycles += solution.dual_cycles
            total_operations += solution.total_operations

        total_makespan = max(
            (
                operation["end_time"]
                for operations in merged_operations.values()
                for operation in operations
            ),
            default=0.0,
        )
        return DualCycleSchedule(
            crane_operations=dict(merged_operations),
            makespan=total_makespan,
            dual_cycles=total_dual_cycles,
            total_operations=total_operations,
            coordination_overhead=total_coordination_overhead,
        )

    def dual_cycle_divide_conquer(
        self, vessel_stacks: List[ContainerStack]
    ) -> DualCycleSchedule:
        """Decompose, sequence, coordinate, and integrate the schedule."""
        print("Starting the divide-and-conquer dual-cycling heuristic...")
        bay_subproblems = self.decompose_by_bays(vessel_stacks)
        print(f"Phase 1: Created {len(bay_subproblems)} bay subproblems")

        bay_solutions = []
        for bay in bay_subproblems:
            if len(bay.stacks) <= self.threshold:
                solution = self.johnsons_rule_dual_cycle(bay)
                print(f"  {bay.id}: Applied Johnson's rule directly")
            else:
                groups = self.partition_stacks(bay)
                print(f"  {bay.id}: Partitioned into {len(groups)} stack groups")
                group_solutions = [
                    self.johnsons_rule_dual_cycle(group) for group in groups
                ]
                solution = self.coordinate_bay_solutions(group_solutions)

            bay_solutions.append(solution)
            print(
                f"    Makespan: {solution.makespan:.1f} min, "
                f"Dual cycles: {solution.dual_cycles}"
            )

        print(f"\nPhase 3: Coordinating {len(bay_solutions)} bay solutions")
        final_schedule = self.coordinate_bay_solutions(bay_solutions)
        return self.optimize_interfaces(final_schedule)

    def optimize_interfaces(
        self, schedule: DualCycleSchedule
    ) -> DualCycleSchedule:
        """Inspect consecutive operations for additional dual-cycle pairs."""
        additional_dual_cycles = 0
        for operations in schedule.crane_operations.values():
            operations.sort(key=lambda operation: operation["start_time"])
            for index in range(len(operations) - 1):
                current = operations[index]
                following = operations[index + 1]
                gap = following["start_time"] - current["end_time"]
                if (
                    current["type"] == "unload"
                    and following["type"] == "load"
                    and 0 <= gap < 5.0
                    and not following.get("dual_cycle", False)
                ):
                    saving = following["duration"] * 0.2
                    following["dual_cycle"] = True
                    following["duration"] -= saving
                    following["end_time"] -= saving
                    for later in operations[index + 2 :]:
                        later["start_time"] -= saving
                        later["end_time"] -= saving
                    additional_dual_cycles += 1

        makespan = max(
            (
                operation["end_time"]
                for operations in schedule.crane_operations.values()
                for operation in operations
            ),
            default=0.0,
        )
        return DualCycleSchedule(
            crane_operations=schedule.crane_operations,
            makespan=makespan,
            dual_cycles=schedule.dual_cycles + additional_dual_cycles,
            total_operations=schedule.total_operations,
            coordination_overhead=schedule.coordination_overhead,
        )


print("Divide-and-conquer algorithm class defined successfully")

### Concrete example

The example has six bays with three stacks per bay. Each stack receives generated import and export containers, and its total unloading and loading times are assigned separately. Container-level processing times sum to the corresponding stack-level times. The generated container counts depend on the fixed random seed; they are reported by the code rather than assumed in advance.

The default threshold is six stacks. Since each paired-bay subproblem contains six stacks, the example applies Johnson's rule directly to each paired-bay group. The hatch-blocked partition is available for groups larger than the threshold.

In [ ]:
solver = DualCycleDivideConquer(
    num_cranes=3,
    num_bays=6,
    stacks_per_bay=3,
    threshold=6,
)

print("=== Problem setup ===")
print(
    f"Configuration: {solver.num_cranes} cranes, {solver.num_bays} bays, "
    f"{len(solver.stacks)} stacks"
)
print(f"Base-case threshold: {solver.threshold} stacks per group")

print("\n=== Bay configuration ===")
for bay in range(1, solver.num_bays + 1):
    bay_stacks = [stack for stack in solver.stacks if stack.bay == bay]
    total_unload = sum(stack.total_unload_time for stack in bay_stacks)
    total_load = sum(stack.total_load_time for stack in bay_stacks)
    total_containers = sum(stack.total_containers for stack in bay_stacks)
    blocked_count = sum(stack.hatch_blocked for stack in bay_stacks)
    print(
        f"Bay {bay}: {len(bay_stacks)} stacks, {total_containers} containers, "
        f"unload {total_unload:.0f} min, load {total_load:.1f} min, "
        f"hatch-blocked stacks {blocked_count}"
    )

In [ ]:
final_schedule = solver.dual_cycle_divide_conquer(solver.stacks)

print("\n=== Final schedule results ===")
print(f"Total makespan: {final_schedule.makespan:.1f} minutes")
print(f"Dual cycles: {final_schedule.dual_cycles}")
print(f"Total operations: {final_schedule.total_operations}")
print(f"Dual-cycle ratio: {final_schedule.dual_cycle_ratio:.1%}")
print(
    f"Coordination delay applied: "
    f"{final_schedule.coordination_overhead:.1f} minutes"
)

print("\n=== Decomposition statistics ===")
for name, value in solver.decomposition_stats.items():
    print(f"{name.replace('_', ' ').title()}: {value}")

### Interpreting the schedule

The **makespan** is the latest operation end time across the crane schedules. Each operation records its start, duration, and end; the generated timeline uses the modeled duration, including the 20% reduction for a paired export load. The **dual-cycle count** counts paired import-export moves, and the **dual-cycle ratio** is \(2D/N\), where \(D\) is the number of pairs and \(N\) is the total number of container operations. Thus the ratio measures the share of operations participating in a pair and lies between zero and one when each operation participates in at most one pair.

Coordination points are detected from overlaps in the schedules before coordination. The coordination heuristic applies a schedule-wide delay to the second crane in each identified pair; the reported coordination delay is the sum of those applied offsets. Because conflicts are identified from the initial pairwise schedules and the model omits detailed crane geometry, this quantity is a heuristic coordination measure, not a proof of conflict-free operation or a separate component to add to the makespan.

In [ ]:
def greedy_baseline(
    stacks: List[ContainerStack], num_cranes: int
) -> DualCycleSchedule:
    """Build a round-robin reference schedule without dual-cycle savings."""
    if num_cranes < 1:
        raise ValueError("The number of cranes must be positive.")

    crane_stacks = defaultdict(list)
    for index, stack in enumerate(stacks):
        crane_stacks[index % num_cranes].append(stack)

    operations = {}
    total_operations = 0
    for crane_id, assigned_stacks in crane_stacks.items():
        crane_operations = []
        current_time = 0.0
        for stack in assigned_stacks:
            for container in stack.import_containers:
                duration = container["unload_time"]
                crane_operations.append(
                    {
                        "type": "unload",
                        "container_id": container["id"],
                        "stack_id": stack.id,
                        "bay": stack.bay,
                        "start_time": current_time,
                        "duration": duration,
                        "end_time": current_time + duration,
                        "dual_cycle": False,
                    }
                )
                current_time += duration
                total_operations += 1

            for container in stack.export_containers:
                duration = container["load_time"]
                crane_operations.append(
                    {
                        "type": "load",
                        "container_id": container["id"],
                        "stack_id": stack.id,
                        "bay": stack.bay,
                        "start_time": current_time,
                        "duration": duration,
                        "end_time": current_time + duration,
                        "dual_cycle": False,
                    }
                )
                current_time += duration
                total_operations += 1
        operations[crane_id] = crane_operations

    makespan = max(
        (op["end_time"] for ops in operations.values() for op in ops),
        default=0.0,
    )
    return DualCycleSchedule(
        crane_operations=operations,
        makespan=makespan,
        dual_cycles=0,
        total_operations=total_operations,
        coordination_overhead=0.0,
    )


baseline_schedule = greedy_baseline(solver.stacks, solver.num_cranes)
if baseline_schedule.makespan > 0:
    makespan_improvement = (
        (baseline_schedule.makespan - final_schedule.makespan)
        / baseline_schedule.makespan
        * 100
    )
else:
    makespan_improvement = 0.0
dual_cycle_improvement = final_schedule.dual_cycles - baseline_schedule.dual_cycles

print("=== Reference comparison ===")
print(
    f"Divide-and-conquer heuristic: {final_schedule.makespan:.1f} min, "
    f"{final_schedule.dual_cycle_ratio:.1%} dual-cycle ratio"
)
print(
    f"Round-robin reference:        {baseline_schedule.makespan:.1f} min, "
    f"{baseline_schedule.dual_cycle_ratio:.1%} dual-cycle ratio"
)
print("\nDifference relative to the reference:")
print(f"  Makespan reduction: {makespan_improvement:.1f}%")
print(f"  Additional dual cycles: {dual_cycle_improvement}")
print(
    f"  Coordination delay applied: "
    f"{final_schedule.coordination_overhead:.1f} minutes"
)

### Visual inspection

The charts compare makespan and dual-cycle ratio for the heuristic and the round-robin reference, show the workload implied by the stack-level processing times, and display a short crane timeline. Timeline colors distinguish paired-cycle operations from single-cycle operations. The comparison is descriptive: the two schedules use different sequencing and cycle assumptions, so it is not an optimality comparison.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

methods = ["Divide-and-conquer", "Round-robin reference"]
makespans = [final_schedule.makespan, baseline_schedule.makespan]
colors = ["#2ecc71", "#e74c3c"]

bars = axes[0, 0].bar(methods, makespans, color=colors, alpha=0.75)
axes[0, 0].set_ylabel("Makespan (minutes)")
axes[0, 0].set_title("Makespan comparison")
axes[0, 0].tick_params(axis="x", rotation=15)
axes[0, 0].grid(True, alpha=0.3, axis="y")
for bar, value in zip(bars, makespans):
    axes[0, 0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.1f}",
        ha="center",
        va="bottom",
    )

ratios = [final_schedule.dual_cycle_ratio, baseline_schedule.dual_cycle_ratio]
bars = axes[0, 1].bar(methods, ratios, color=colors, alpha=0.75)
axes[0, 1].set_ylabel("Share of operations in dual cycles")
axes[0, 1].set_ylim(0, 1.05)
axes[0, 1].set_title("Dual-cycle ratio")
axes[0, 1].tick_params(axis="x", rotation=15)
axes[0, 1].grid(True, alpha=0.3, axis="y")
for bar, value in zip(bars, ratios):
    axes[0, 1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.1%}",
        ha="center",
        va="bottom",
    )

bay_workloads = []
for bay in range(1, solver.num_bays + 1):
    bay_stacks = [stack for stack in solver.stacks if stack.bay == bay]
    bay_workloads.append(
        sum(stack.total_unload_time + stack.total_load_time for stack in bay_stacks)
    )
axes[0, 2].bar(range(1, solver.num_bays + 1), bay_workloads, color="#9b59b6")
axes[0, 2].set_xlabel("Bay number")
axes[0, 2].set_ylabel("Nominal processing workload (minutes)")
axes[0, 2].set_title("Bay workload")
axes[0, 2].grid(True, alpha=0.3, axis="y")

timeline_axis = axes[1, 0]
timeline_operations = final_schedule.crane_operations.get(0, [])[:10]
for row, operation in enumerate(timeline_operations):
    color = "#2ecc71" if operation.get("dual_cycle", False) else "#3498db"
    timeline_axis.barh(
        row,
        operation["duration"],
        left=operation["start_time"],
        color=color,
        alpha=0.8,
        height=0.6,
    )
    timeline_axis.text(
        operation["start_time"] + operation["duration"] / 2,
        row,
        operation["type"].upper(),
        ha="center",
        va="center",
        fontsize=8,
    )
timeline_axis.set_xlabel("Time (minutes)")
timeline_axis.set_ylabel("Operation index")
timeline_axis.set_title("Crane 0 timeline (first 10 operations)")
timeline_axis.grid(True, alpha=0.3)
timeline_axis.legend(
    handles=[
        plt.Rectangle((0, 0), 1, 1, color="#2ecc71", label="Dual-cycle operation"),
        plt.Rectangle((0, 0), 1, 1, color="#3498db", label="Single-cycle operation"),
    ],
    loc="best",
)

stat_names = ["Subproblems", "Coordination points", "Johnson applications"]
stat_values = [
    solver.decomposition_stats["subproblems_generated"],
    solver.decomposition_stats["coordination_points"],
    solver.decomposition_stats["johnson_applications"],
]
bars = axes[1, 1].bar(stat_names, stat_values, color="#f39c12")
axes[1, 1].set_ylabel("Count")
axes[1, 1].set_title("Decomposition statistics")
axes[1, 1].tick_params(axis="x", rotation=15)
axes[1, 1].grid(True, alpha=0.3, axis="y")
for bar, value in zip(bars, stat_values):
    axes[1, 1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        str(value),
        ha="center",
        va="bottom",
    )

metric_names = ["Makespan\nreduction (%)", "Dual-cycle\nratio (%)", "Coordination\ndelay (min)"]
metric_values = [
    makespan_improvement,
    final_schedule.dual_cycle_ratio * 100,
    final_schedule.coordination_overhead,
]
metric_colors = ["#27ae60", "#3498db", "#e67e22"]
bars = axes[1, 2].bar(metric_names, metric_values, color=metric_colors)
axes[1, 2].set_ylabel("Value (units shown on labels)")
axes[1, 2].set_title("Summary measures")
axes[1, 2].grid(True, alpha=0.3, axis="y")
for bar, value in zip(bars, metric_values):
    axes[1, 2].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.1f}",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()

### Results summary and interpretation

The printed and plotted values are computed from the generated example when the notebook is run. They should be interpreted in light of the model assumptions:

- The paired-bay decomposition and Johnson ordering define the local sequencing stage.
- A dual cycle is counted only when an import and an export in the same stack are paired, or when the interface inspection identifies a consecutive unload-load pair.
- The makespan reflects the operation durations and schedule-wide coordination delays encoded in this heuristic.
- The round-robin schedule is a reference, not a controlled experiment that isolates the effect of a single modeling choice.
- The coordination-point count and accumulated delay describe the heuristic's response to detected overlaps; they do not establish physical feasibility.

A smaller makespan is preferable under this model, while a larger dual-cycle ratio indicates that a larger share of moves participates in paired cycles. These measures need not improve together. This example demonstrates the decomposition and scheduling sequence; it does not establish scalability or real-time suitability for larger terminal instances.

In [ ]:
total_containers = sum(stack.total_containers for stack in solver.stacks)
expected_operations = sum(
    len(stack.import_containers) + len(stack.export_containers)
    for stack in solver.stacks
)
reported_operations = sum(
    len(operations)
    for operations in final_schedule.crane_operations.values()
)

assert reported_operations == expected_operations
assert final_schedule.total_operations == expected_operations
assert 0.0 <= final_schedule.dual_cycle_ratio <= 1.0
assert all(
    np.isclose(operation["end_time"] - operation["start_time"], operation["duration"])
    for operations in final_schedule.crane_operations.values()
    for operation in operations
)

print("=" * 68)
print("DUAL-CYCLING QUAY CRANE PROBLEM — TIER 2 SUMMARY")
print("=" * 68)
print(
    f"Problem size: {solver.num_cranes} cranes, {solver.num_bays} bays, "
    f"{len(solver.stacks)} stacks"
)
print(f"Total containers and operations: {total_containers}")
print(f"Decomposition threshold: {solver.threshold} stacks per group")
print("\nHeuristic schedule:")
print(f"  Makespan: {final_schedule.makespan:.1f} minutes")
print(f"  Dual cycles: {final_schedule.dual_cycles}")
print(f"  Dual-cycle ratio: {final_schedule.dual_cycle_ratio:.1%}")
print(f"  Total operations: {final_schedule.total_operations}")
print("\nDecomposition and coordination:")
print(
    f"  Subproblems generated: "
    f"{solver.decomposition_stats['subproblems_generated']}"
)
print(
    f"  Detected coordination points: "
    f"{solver.decomposition_stats['coordination_points']}"
)
print(
    f"  Johnson rule applications: "
    f"{solver.decomposition_stats['johnson_applications']}"
)
print(f"  Coordination delay applied: {final_schedule.coordination_overhead:.1f} minutes")
print("\nReference comparison:")
print(f"  Round-robin makespan: {baseline_schedule.makespan:.1f} minutes")
print(f"  Makespan reduction: {makespan_improvement:.1f}%")
print("\nSchedule consistency checks passed.")
print("=" * 68)